In [ ]:
只适用于D,W

In [1]:
import os
import pandas as pd
import numpy as np
from itertools import product
from tqdm import tqdm

base = r"C:\Users\wrz\Desktop\FYPProject"
data_dir = os.path.join(base, "Data")

industry_file_map = {
    "化工": "159870.OF.csv",
    "传统能源": "159930.OF.csv",
    "金融": "510230.OF.csv",
    "医药": "512010.OF.csv",
    "消费": "159928.OF.csv",
    "房地产": "512200.OF.csv",
    "电动车新能源": "515030.OF.csv",
    "高端装备": "516320.OF.csv",
    "旅游": "159766.OF.csv",
    "半导体与信息技术": "159939.OF.csv"
}


FREQ = "2W"  


custom_2w_periods = pd.to_datetime([
    "2023-12-01","2023-12-16",
    "2024-01-01", "2024-01-16", "2024-02-01", "2024-02-16", "2024-03-01", "2024-03-16",
    "2024-04-01", "2024-04-16", "2024-05-01", "2024-05-16", "2024-06-01", "2024-06-16",
    "2024-07-01", "2024-07-16", "2024-08-01", "2024-08-16", "2024-09-01", "2024-09-16",
    "2024-10-01", "2024-10-16", "2024-11-01", "2024-11-16", "2024-12-01", "2024-12-16"
])


def get_factor_file(name, freq):
    return os.path.join(base, f"{name}_{freq}.csv")


sentiment_file = get_factor_file("adjusted_policy_sentiment_score", FREQ)
market_file = get_factor_file("market_sentiment_factor", FREQ)

sentiment = pd.read_csv(sentiment_file, parse_dates=["period"])
market = pd.read_csv(market_file, parse_dates=["period"])

if FREQ == "2W":
    sentiment["period"] = pd.to_datetime(sentiment["period"])
    market["period"] = pd.to_datetime(market["period"])

    sentiment_bins = pd.cut(sentiment["period"], bins=custom_2w_periods, right=False)
    sentiment_grouped = sentiment.groupby(["industry", sentiment_bins]).mean(numeric_only=True).reset_index()
    sentiment_grouped["period"] = sentiment_grouped["period"].apply(lambda x: x.left)

    market_bins = pd.cut(market["period"], bins=custom_2w_periods, right=False)
    market_grouped = market.groupby(market_bins).mean(numeric_only=True).reset_index()
    market_grouped["period"] = market_grouped["period"].apply(lambda x: x.left)

    sentiment = sentiment_grouped
    market = market_grouped

else:
    sentiment["period"] = pd.to_datetime(sentiment["period"])
    market["period"] = pd.to_datetime(market["period"])

    if FREQ != "M":
        sentiment = sentiment.groupby(["industry", pd.Grouper(key="period", freq=FREQ)]).mean(numeric_only=True).reset_index()
        market = market.groupby(pd.Grouper(key="period", freq=FREQ)).mean(numeric_only=True).reset_index()

factors = sentiment.merge(market, on="period")

returns_dict = {}
for industry, fn in industry_file_map.items():
    df = pd.read_csv(os.path.join(data_dir, fn), parse_dates=["日期"])
    df.set_index("日期", inplace=True)

    if FREQ == "2W":
        closes = df["收盘价(元)"]
        rets = []
        for i in range(1, len(custom_2w_periods)):
            start = custom_2w_periods[i - 1]
            end = custom_2w_periods[i]

            try:
                price_start = closes[:start].iloc[-1]
                price_end = closes[:end].iloc[-1]
                ret = (price_end - price_start) / price_start
                rets.append((end, ret))
            except:
                continue

        returns_dict[industry] = pd.Series(
            [r[1] for r in rets],
            index=[r[0] for r in rets]
        )

    else:
        price_resampled = df["收盘价(元)"].resample(FREQ).last()
        ret = price_resampled.pct_change().dropna()
        returns_dict[industry] = ret

returns_df = pd.DataFrame(returns_dict)
returns_df.index.name = "period"

factors["period"] = pd.to_datetime(factors["period"])
returns_df.index = pd.to_datetime(returns_df.index)

common_periods = factors["period"].unique()
returns_df = returns_df.reindex(common_periods).dropna(how="all")

returns_long = returns_df.reset_index().melt(id_vars="period", var_name="industry", value_name="return")
factors_full = factors.merge(returns_long, on=["period", "industry"], how="left")


output_file = f"factors_with_returns_{FREQ}.csv"
output_path = os.path.join(base, output_file)
factors_full.to_csv(output_path, index=False)

print(f" 粒度 {FREQ} 整合完成，文件保存至：{output_path}")


 粒度 2W 整合完成，文件保存至：C:\Users\wrz\Desktop\FYPProject\factors_with_returns_2W.csv


In [ ]:
只适用于M

In [2]:
import os
import pandas as pd
import numpy as np
from itertools import product
from tqdm import tqdm

base = r"C:\Users\wrz\Desktop\FYPProject"
data_dir = os.path.join(base, "Data")

industry_file_map = {
    "化工": "159870.OF.csv",
    "传统能源": "159930.OF.csv",
    "金融": "510230.OF.csv",
    "医药": "512010.OF.csv",
    "消费": "159928.OF.csv",
    "房地产": "512200.OF.csv",
    "电动车新能源": "515030.OF.csv",
    "高端装备": "516320.OF.csv",
    "旅游": "159766.OF.csv",
    "半导体与信息技术": "159939.OF.csv"
}


sentiment = pd.read_csv(os.path.join(base, "adjusted_policy_sentiment_score_M.csv"),
                        parse_dates=["period"])
market    = pd.read_csv(os.path.join(base, "market_sentiment_factor_M.csv"),
                        parse_dates=["period"])
factors   = sentiment.merge(market, on="period")  

monthly_returns = {}
for industry, fn in industry_file_map.items():
    df = pd.read_csv(os.path.join(data_dir, fn), parse_dates=["日期"])
    df.set_index("日期", inplace=True)
    mclose = df["收盘价(元)"].resample("M").last()
    mret   = mclose.pct_change().dropna()
    monthly_returns[industry] = mret

monthly_returns = pd.DataFrame(monthly_returns)
monthly_returns.index.name = "period"


factors['period'] = pd.to_datetime(factors['period']).dt.to_period('M')
monthly_returns.index = monthly_returns.index.to_period('M')

factors = factors[(factors['period'] >= '2024-01') & (factors['period'] <= '2024-12')]
monthly_returns = monthly_returns.loc[(monthly_returns.index >= '2023-12') & (monthly_returns.index <= '2024-12')]


common_periods = factors["period"].unique()
monthly_returns = monthly_returns.reindex(common_periods).dropna(how="all")


returns_long = monthly_returns.reset_index().melt(id_vars="period", var_name="industry", value_name="return")


factors_full = factors.merge(returns_long, on=["period", "industry"], how="left")


output_path = os.path.join(base, "factors_with_returns_M.csv")
factors_full.to_csv(output_path, index=False)

print(f"已保存至：{output_path}")

已保存至：C:\Users\wrz\Desktop\FYPProject\factors_with_returns_M.csv


In [ ]:
只适用于2W

In [3]:
import pandas as pd


file_path = r'C:\Users\wrz\Desktop\FYPProject\factors_with_returns_D.csv'
df = pd.read_csv(file_path, parse_dates=["period"])


s_adjusted_file_path = r'C:\Users\wrz\Desktop\FYPProject\adjusted_policy_sentiment_score_2W.csv'
df_s_adjusted = pd.read_csv(s_adjusted_file_path, parse_dates=["period"])


market_sentiment_file_path = r'C:\Users\wrz\Desktop\FYPProject\market_sentiment_factor_2W.csv'
df_m = pd.read_csv(market_sentiment_file_path, parse_dates=["period"])


custom_2w_periods = pd.to_datetime([
    "2024-01-01", "2024-01-16", "2024-02-01", "2024-02-16", "2024-03-01", "2024-03-16",
    "2024-04-01", "2024-04-16", "2024-05-01", "2024-05-16", "2024-06-01", "2024-06-16",
    "2024-07-01", "2024-07-16", "2024-08-01", "2024-08-16", "2024-09-01", "2024-09-16",
    "2024-10-01", "2024-10-16", "2024-11-01", "2024-11-16", "2024-12-01", "2024-12-16"
])


df["period"] = pd.to_datetime(df["period"])
df_s_adjusted["period"] = pd.to_datetime(df_s_adjusted["period"])
df_m["period"] = pd.to_datetime(df_m["period"])


bins = pd.cut(df["period"], bins=custom_2w_periods, right=False)


df["bin"] = bins
df_grouped = df.groupby(["industry", "bin"]).agg({
    "return": lambda x: (1 + x).prod() - 1 
}).reset_index()


df_grouped["period"] = df_grouped["bin"].apply(lambda x: x.left)
df_grouped["period"] = pd.to_datetime(df_grouped["period"])  


df_s_adjusted["period"] = pd.to_datetime(df_s_adjusted["period"])
df_m["period"] = pd.to_datetime(df_m["period"])


df_grouped = pd.merge(df_grouped, df_s_adjusted, on=["period", "industry"], how="left")
df_grouped = pd.merge(df_grouped, df_m, on="period", how="left")


df_grouped = df_grouped[["industry", "period", "S_adjusted", "M", "return"]]
output_file_path = r'C:\Users\wrz\Desktop\FYPProject\factors_with_returns_2W.csv'
df_grouped.to_csv(output_file_path, index=False)

print(f"文件已保存至：{output_file_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\wrz\\Desktop\\FYPProject\\factors_with_returns_D.csv'